<a href="https://colab.research.google.com/github/0szgn0/DataForge-agri-yield-prediction/blob/main/dataforge_kodu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install catboost
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import warnings

warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    drive.mount('/content/drive')
except:
    print("Colab ortamında değilsin veya Drive zaten bağlı.")

print("\n[1/5] Veriler yükleniyor ve temizleniyor...")
train = pd.read_csv('/content/drive/MyDrive/train.csv')
test = pd.read_csv('/content/drive/MyDrive/test.csv')

train = train.dropna(subset=['Production']).reset_index(drop=True)

test_ids = test['ID']
test_features = test.drop(columns=['ID'])

for df in [train, test_features]:
    df['Crop'] = df['Crop'].fillna(df['Crop'].mode()[0])
    df['Season'] = df['Season'].fillna(df['Season'].mode()[0])
    df['Crop_Value_Index'] = df['Crop_Value_Index'].fillna(df['Crop_Value_Index'].median())

def extract_year(year_str):
    if pd.isna(year_str): return -1
    return int(str(year_str).split('-')[0])

train['Year'] = train['Year'].apply(extract_year)
test_features['Year'] = test_features['Year'].apply(extract_year)

print("Target Encoding, Log dönüşümleri")

def advanced_feature_engineering(train_df, test_df):
    train_feat = train_df.copy()
    test_feat = test_df.copy()

    for df in [train_feat, test_feat]:
        df['Log_Area'] = np.log1p(df['Area'])
        df['Log_Potential_Yield'] = np.log1p(df['Area'] * df['Crop_Value_Index'])
        df['Crop_x_Unit'] = df['Crop'].astype(str) + "_" + df['Production Units'].astype(str)

    _, bins = pd.qcut(train_feat['Area'], q=4, retbins=True, labels=False, duplicates='drop')
    bins[0], bins[-1] = -np.inf, np.inf

    train_feat['Area_Bin'] = pd.cut(train_feat['Area'], bins=bins, labels=['Kucuk', 'Orta', 'Buyuk', 'Cok_Buyuk']).astype(str)
    test_feat['Area_Bin'] = pd.cut(test_feat['Area'], bins=bins, labels=['Kucuk', 'Orta', 'Buyuk', 'Cok_Buyuk']).astype(str)

    train_feat['Temp_Log_Target'] = np.log1p(train_feat['Production'])

    crop_mean = train_feat.groupby('Crop')['Temp_Log_Target'].mean().to_dict()
    state_mean = train_feat.groupby('State')['Temp_Log_Target'].mean().to_dict()

    for df in [train_feat, test_feat]:
        df['Mean_Log_Prod_by_Crop'] = df['Crop'].map(crop_mean)
        df['Mean_Log_Prod_by_State'] = df['State'].map(state_mean)

    global_mean = train_feat['Temp_Log_Target'].mean()
    test_feat['Mean_Log_Prod_by_Crop'].fillna(global_mean, inplace=True)
    test_feat['Mean_Log_Prod_by_State'].fillna(global_mean, inplace=True)

    train_feat.drop(columns=['Temp_Log_Target'], inplace=True)

    return train_feat, test_feat

train, test_features = advanced_feature_engineering(train, test_features)


print(K-Fold modeli için veriler hazırlanıyor")

cat_features = ['State', 'District', 'Crop', 'Season', 'Area Units', 'Production Units', 'Crop_x_Unit', 'Area_Bin']

X = train.drop(columns=['Production'])
y = np.log1p(train['Production']) # Hedef değişkenin logaritmasını alıyoruz

# Algoritma hata vermesin diye tüm kategorik sütunları String yapıyoruz
for col in cat_features:
    X[col] = X[col].fillna('Unknown').astype(str)
    test_features[col] = test_features[col].fillna('Unknown').astype(str)

# 5. K-FOLD CROSS VALIDATION İLE CATBOOST EĞİTİMİ
print("[4/5] 5-Fold CatBoost Eğitimi başlıyor...\n")

kf = KFold(n_splits=5, shuffle=True, random_state=42)

test_preds = np.zeros(len(test_features))
cv_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    print(f"--- FOLD {fold+1} ---")

    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

    model = CatBoostRegressor(
        iterations=1500,
        learning_rate=0.05,
        depth=7,
        l2_leaf_reg=3,
        cat_features=cat_features,
        eval_metric='RMSE',
        random_seed=42,
        verbose=300
    )

    model.fit(
        X_tr, y_tr,
        eval_set=(X_va, y_va),
        early_stopping_rounds=100,
        verbose_eval=300
    )

    # Logaritmayı tersine çevirip gerçek RMSE hesaplama
    val_preds_log = model.predict(X_va)
    fold_rmse = np.sqrt(mean_squared_error(np.expm1(y_va), np.expm1(val_preds_log)))
    cv_scores.append(fold_rmse)
    print(f"Fold {fold+1} Gerçek Ölçekte RMSE: {fold_rmse:.2f}\n")

    # Test setini tahmin et ve topla (En son 5'e böleceğiz)
    test_preds += model.predict(test_features) / kf.n_splits

print(f"🌟 SÜREÇ TAMAMLANDI! Ortalama K-Fold CV RMSE: {np.mean(cv_scores):.2f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

[1/5] Veriler yükleniyor ve temizleniyor...
[2/5] Yeni özellikler türetiliyor (Target Encoding, Log dönüşümleri)...
[3/5] K-Fold modeli için veriler hazırlanıyor...
[4/5] 5-Fold CatBoost Eğitimi başlıyor...

--- FOLD 1 ---
0:	learn: 3.3681456	test: 3.3506738	best: 3.3506738 (0)	total: 419ms	remaining: 10m 28s
300:	learn: 1.3490461	test: 1.3605141	best: 1.3605039 (285)	total: 1m 1s	remaining: 4m 4s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 1.360470615
bestIteration = 304

Shrink model to first 305 iterations.
Fold 1 Gerçek Ölçekte RMSE: 6309832.01

--- FOLD 2 ---
0:	learn: 3.3665660	test: 3.3577941	best: 3.3577941 (0)	total: 334ms	remaining: 8m 21s
300:	learn: 1.3491127	test: 1.3587800	best: 1.3587800 (300)	total: 1m 2s	remaining: 4m 9s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 1.35856716
bestIteration = 423



In [ ]:
import pandas as pd
import numpy as np

print("--- ZAMAN MAKİNESİ: LAG (GECİKME) DEĞİŞKENLERİ EKLENİYOR ---")

def add_lag_features(train_df, test_df):
    # Orijinal verileri bozmamak için kopyalayalım
    df_train = train_df.copy()
    df_test = test_df.copy()

    # Train ve Test'i birleştirip tekrar ayırabilmek için işaretleyelim
    df_train['is_test'] = 0
    df_test['is_test'] = 1

    # Test setinde 'Production' sütunu yok, hata vermemesi için boş (NaN) olarak ekleyelim
    df_test['Production'] = np.nan

    # Verileri alt alta birleştirelim
    df_all = pd.concat([df_train, df_test], axis=0, ignore_index=True)

    # Hedef değişkenin logaritmasını alalım (Aşırı uçuk üretim sayıları modeli bozmasın diye)
    df_all['Target_Log'] = np.log1p(df_all['Production'])

    # 1. ADIM: VERİYİ ZAMANA GÖRE SIRALAMAK (EN KRİTİK KISIM!)
    # Aynı yerdeki aynı ürünün yıllar içindeki akışını görmek için sıralıyoruz
    df_all = df_all.sort_values(by=['State', 'District', 'Crop', 'Season', 'Year'])

    # Hangi gruplar içinde geçmişe bakacağız?
    groupby_cols = ['State', 'District', 'Crop', 'Season']

    # 2. ADIM: SHIFT (KAYDIRMA) İŞLEMİ
    # shift(1) bir önceki satırı (yani bir önceki yılı) getirir
    df_all['Lag_1_Log_Prod'] = df_all.groupby(groupby_cols)['Target_Log'].shift(1)
    df_all['Lag_2_Log_Prod'] = df_all.groupby(groupby_cols)['Target_Log'].shift(2)

    # İlk yıllar için geçmiş veri olmayacağından NaN çıkacaktır. Bunları 0 ile dolduralım.
    df_all['Lag_1_Log_Prod'].fillna(0, inplace=True)
    df_all['Lag_2_Log_Prod'].fillna(0, inplace=True)

    # 3. ADIM: TRAIN VE TEST'İ GERİ AYIRMA
    # is_test etiketini kullanarak veriyi ilk haline bölelim
    train_new = df_all[df_all['is_test'] == 0].copy()
    test_new = df_all[df_all['is_test'] == 1].copy()

    # İşimizi bitiren geçici sütunları temizleyelim
    train_new.drop(columns=['is_test', 'Target_Log'], inplace=True)
    test_new.drop(columns=['is_test', 'Target_Log', 'Production'], inplace=True)

    # Train setini orijinal indeks sıralamasına geri getirelim ki K-Fold CV bozulmasın
    train_new = train_new.sort_index()
    test_new = test_new.sort_index()

    return train_new, test_new

# Fonksiyonu çalıştırıp verilerimizi güncelleyelim
train, test_features = add_lag_features(train, test_features)

print("Başarılı! Yeni eklenen sütunlar:")
print(train[['State', 'Year', 'Crop', 'Production', 'Lag_1_Log_Prod', 'Lag_2_Log_Prod']].head())

--- ZAMAN MAKİNESİ: LAG (GECİKME) DEĞİŞKENLERİ EKLENİYOR ---
Başarılı! Yeni eklenen sütunlar:
             State  Year           Crop    Production  Lag_1_Log_Prod  \
0            Konya  2019          Mısır  5.566640e+02        0.000000   
1          Trabzon  2003         Fındık  6.565997e+07       18.000000   
2            Konya  2015        Kuşyemi  2.685375e+04        9.427536   
3            Adana  2017  Kırmızı Biber  2.866840e+02        4.011995   
4  Ankara (Merkez)  2000           Darı  4.056303e+03        5.808052   

   Lag_2_Log_Prod  
0        3.161840  
1        0.000000  
2        7.367575  
3        1.041336  
4        6.959964  


In [ ]:
from sklearn.metrics import mean_absolute_error # MSE yerine MAE'yi import ediyoruz
import numpy as np

print("[4/5] 5-Fold CatBoost Eğitimi başlıyor (MAE Odaklı)...\n")

kf = KFold(n_splits=5, shuffle=True, random_state=42)

test_preds = np.zeros(len(test_features))
cv_scores_mae = [] # Liste adını MAE'ye uygun güncelledik

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    print(f"--- FOLD {fold+1} ---")

    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

    model = CatBoostRegressor(
        iterations=1500,
        learning_rate=0.05,
        depth=7,
        l2_leaf_reg=3,
        cat_features=cat_features,
        loss_function='MAE',
        eval_metric='MAE',
        random_seed=42,
        verbose=300
    )

    model.fit(
        X_tr, y_tr,
        eval_set=(X_va, y_va),
        early_stopping_rounds=100,
        verbose_eval=300
    )

    # Tahminleri al (Logaritmik tahminler dönecek)
    val_preds_log = model.predict(X_va)

    # <-- DEĞİŞTİ: Logaritmayı tersine çevirip (expm1) doğrudan MAE hesaplıyoruz (Karekök - sqrt - yok)
    fold_mae = mean_absolute_error(np.expm1(y_va), np.expm1(val_preds_log))
    cv_scores_mae.append(fold_mae)
    print(f"Fold {fold+1} Gerçek Ölçekte MAE: {fold_mae:.2f}\n")

    # Test setini tahmin et ve topla (En son 5'e böleceğiz)
    test_preds += model.predict(test_features) / kf.n_splits

print(f"Ortalama K-Fold CV MAE: {np.mean(cv_scores_mae):.2f}")

[4/5] 5-Fold CatBoost Eğitimi başlıyor (MAE Odaklı)...

--- FOLD 1 ---
0:	learn: 2.6083884	test: 2.5947224	best: 2.5947224 (0)	total: 222ms	remaining: 5m 32s
300:	learn: 0.9182542	test: 0.9256338	best: 0.9256338 (300)	total: 1m 8s	remaining: 4m 33s
600:	learn: 0.9115790	test: 0.9240255	best: 0.9240177 (599)	total: 2m 15s	remaining: 3m 22s
900:	learn: 0.9078675	test: 0.9239151	best: 0.9239116 (899)	total: 3m 17s	remaining: 2m 11s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.9239115621
bestIteration = 899

Shrink model to first 900 iterations.
Fold 1 Gerçek Ölçekte MAE: 803264.50

--- FOLD 2 ---
0:	learn: 2.6091140	test: 2.5940598	best: 2.5940598 (0)	total: 340ms	remaining: 8m 29s


KeyboardInterrupt: 

In [ ]:
# Üretimin en üst %1'lik kısmını (devasa aykırı değerleri) bul
ust_sinir = train['Production'].quantile(0.99)

# Bu değerden büyük olanları, üst sınıra eşitle (Silmiyoruz, sadece törpülüyoruz)
train['Production'] = np.clip(train['Production'], a_min=None, a_max=ust_sinir)

# Ardından y = np.log1p(train['Production']) adımına geçebilirsin.

In [ ]:
!pip install lightgbm

In [ ]:
# 1. GRUBA ÖZEL TIRAŞLAMA (GROUP-WISE CLIPPING)
# Orijinal üretim değerlerini bozmadan yeni bir kolon açalım
train['Production_Clipped'] = train['Production'].copy()

# Her 'Crop_x_Unit' grubu için %99'luk üst sınırı bul ve sadece o gruba uygula
for grup in train['Crop_x_Unit'].unique():
    grup_maskesi = train['Crop_x_Unit'] == grup
    ust_sinir = train.loc[grup_maskesi, 'Production'].quantile(0.99)

    # Eğer o grupta yeterli veri yoksa ust_sinir NaN dönebilir, bunu kontrol edelim
    if pd.notna(ust_sinir):
        train.loc[grup_maskesi, 'Production_Clipped'] = np.clip(
            train.loc[grup_maskesi, 'Production'],
            a_min=None,
            a_max=ust_sinir
        )

# 2. TARİHSEL VERİM (YIELD) VE BEKLENEN ÜRETİM
# Train setinde hektar başına düşen ortalama üretimi bulalım
train['Birim_Verim'] = train['Production_Clipped'] / train['Area']

# Ürün ve Şehir bazında ortalama verimi hesaplayalım
verim_sozlugu = train.groupby(['State', 'Crop'])['Birim_Verim'].mean().to_dict()

# Bu ortalama verimi hem Train hem Test setine eşleyelim (Map)
train['Ortalama_Verim'] = train.set_index(['State', 'Crop']).index.map(verim_sozlugu.get)
test_features['Ortalama_Verim'] = test_features.set_index(['State', 'Crop']).index.map(verim_sozlugu.get)

# Test setinde daha önce hiç görülmemiş (NaN) Şehir-Ürün kombinasyonları varsa,
# sadece "Ürün" bazındaki genel ortalama verimle dolduralım
genel_urun_verimi = train.groupby('Crop')['Birim_Verim'].mean().to_dict()
train['Ortalama_Verim'] = train['Ortalama_Verim'].fillna(train['Crop'].map(genel_urun_verimi))
test_features['Ortalama_Verim'] = test_features['Ortalama_Verim'].fillna(test_features['Crop'].map(genel_urun_verimi))

# Son olarak en güçlü yeni sütunumuzu yaratalım: BEKLENEN ÜRETİM
train['Beklenen_Uretim'] = train['Area'] * train['Ortalama_Verim']
test_features['Beklenen_Uretim'] = test_features['Area'] * test_features['Ortalama_Verim']

# İşimiz biten geçici kolonları temizleyelim
train.drop(columns=['Birim_Verim', 'Ortalama_Verim'], inplace=True)
test_features.drop(columns=['Ortalama_Verim'], inplace=True)

print("✅ Gruba özel tıraşlama yapıldı ve 'Beklenen_Uretim' kolonu eklendi!")

--- 🛠️ İLERİ SEVİYE VERİ MÜDAHALESİ BAŞLIYOR 🛠️ ---
✅ Gruba özel tıraşlama yapıldı ve 'Beklenen_Uretim' kolonu eklendi!


In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

print("--- ⚔️ CATBOOST & LIGHTGBM ENSEMBLE BAŞLIYOR ⚔️ ---")

# 1. TIRAŞLAMA VE LOGARİTMA (Senin kaldığın yer)
ust_sinir = train['Production'].quantile(0.99)
train['Production_Clipped'] = np.clip(train['Production'], a_min=None, a_max=ust_sinir)

X = train.drop(columns=['Production', 'Production_Clipped'])
y = np.log1p(train['Production_Clipped'])

# 2. LIGHTGBM İÇİN VERİ HAZIRLIĞI
# LightGBM, kategorik değişkenleri string olarak değil, 'category' tipi olarak ister.
X_lgb = X.copy()
test_features_lgb = test_features.copy()

for col in cat_features:
    X_lgb[col] = X_lgb[col].astype('category')
    test_features_lgb[col] = test_features_lgb[col].astype('category')

# CatBoost için string kalmalı
for col in cat_features:
    X[col] = X[col].astype(str)
    test_features[col] = test_features[col].astype(str)

# 3. K-FOLD EĞİTİM DÖNGÜSÜ
kf = KFold(n_splits=5, shuffle=True, random_state=42)

test_preds_cat = np.zeros(len(test_features))
test_preds_lgb = np.zeros(len(test_features))
cv_scores_mae = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    print(f"\n--- FOLD {fold+1} ---")

    # CatBoost Verileri
    X_tr_cat, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va_cat, y_va = X.iloc[val_idx], y.iloc[val_idx]

    # LightGBM Verileri
    X_tr_lgb = X_lgb.iloc[train_idx]
    X_va_lgb = X_lgb.iloc[val_idx]

    # MODEL 1: CATBOOST
    print("1. CatBoost eğitiliyor...")
    model_cat = CatBoostRegressor(
        iterations=1500, learning_rate=0.05, depth=7,
        cat_features=cat_features, loss_function='MAE',
        verbose=0, random_seed=42 # Log ekranını kirletmemesi için 0 yaptık
    )
    model_cat.fit(X_tr_cat, y_tr, eval_set=(X_va_cat, y_va), early_stopping_rounds=100)
    val_preds_cat = model_cat.predict(X_va_cat)

    # MODEL 2: LIGHTGBM
    print("2. LightGBM eğitiliyor...")
    model_lgb = lgb.LGBMRegressor(
        n_estimators=1500, learning_rate=0.05, max_depth=7,
        objective='mae', random_state=42, n_jobs=-1, verbose=-1
    )
    callbacks = [lgb.early_stopping(100, verbose=False)]
    model_lgb.fit(X_tr_lgb, y_tr, eval_set=[(X_va_lgb, y_va)], callbacks=callbacks)
    val_preds_lgb = model_lgb.predict(X_va_lgb)

    # (ENSEMBLE - %50 CatBoost + %50 LightGBM)
    val_preds_log_ensemble = (val_preds_cat * 0.5) + (val_preds_lgb * 0.5)

    # Logaritmayı tersine çevirip MAE hesaplama
    fold_mae = mean_absolute_error(np.expm1(y_va), np.expm1(val_preds_log_ensemble))
    cv_scores_mae.append(fold_mae)
    print(f"👉 Ensemble Gerçek Ölçekte MAE: {fold_mae:.2f}")

    # Test setlerini tahmin et
    test_preds_cat += model_cat.predict(test_features) / kf.n_splits
    test_preds_lgb += model_lgb.predict(test_features_lgb) / kf.n_splits

print(f"\n🌟 SÜREÇ TAMAMLANDI! Ensemble Ortalama K-Fold MAE: {np.mean(cv_scores_mae):.2f}")

# 4. NİHAİ TEST TAHMİNLERİNİ BİRLEŞTİRME VE KAYDETME
print("Teslimat dosyası oluşturuluyor...")

# Nihai test tahminlerinin de ortalamasını alıyoruz
final_test_preds_log = (test_preds_cat * 0.5) + (test_preds_lgb * 0.5)

# Ters logaritma ile orijinal üretime dön (Negatifleri 0 yap)
final_test_preds = np.expm1(final_test_preds_log)
final_test_preds = np.clip(final_test_preds, a_min=0, a_max=None)

submission = pd.DataFrame({
    'ID': test_ids,
    'Production': final_test_preds
})

submission_path = '/content/drive/MyDrive/submission_ensemble_v1.csv'
submission.to_csv(submission_path, index=False)
print(f"dosyan: {submission_path}")

ModuleNotFoundError: No module named 'catboost'